# SpaceX API Data Collection

**Author:** Aung Paing Soe


## Objective

Collect past launches from the SpaceX REST API, normalize nested JSON, enrich ID fields, and retain Falcon 9 records.


In [ ]:
import requests, pandas as pd
from pandas import json_normalize
launches = requests.get('https://api.spacexdata.com/v4/launches/past', timeout=30).json()
raw = json_normalize(launches)
raw.head()


## Enrich records

For each launch, follow rocket, payload, launchpad, core, and landpad IDs to collect booster, payload mass, orbit, site coordinates, reuse, hardware, and landing outcome.


In [ ]:
def get_json(url):
    r = requests.get(url, timeout=30); r.raise_for_status(); return r.json()
rows=[]
for x in launches:
    rocket=get_json('https://api.spacexdata.com/v4/rockets/'+x['rocket'])
    if rocket.get('name')!='Falcon 9': continue
    pad=get_json('https://api.spacexdata.com/v4/launchpads/'+x['launchpad'])
    payload=get_json('https://api.spacexdata.com/v4/payloads/'+x['payloads'][0]) if x.get('payloads') else {}
    core=x.get('cores',[{}])[0]
    rows.append({'FlightNumber':x.get('flight_number'),'Date':x.get('date_utc'),'PayloadMass':payload.get('mass_kg'),'Orbit':payload.get('orbit'),'LaunchSite':pad.get('name'),'Latitude':pad.get('latitude'),'Longitude':pad.get('longitude'),'Outcome':core.get('landing_success'),'Flights':core.get('flight'),'GridFins':core.get('gridfins'),'Reused':core.get('reused'),'Legs':core.get('legs')})
api_data=pd.DataFrame(rows)
api_data.head()
